In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType
import time

In [3]:
spark = SparkSession.builder.appName("AdvSparkApp").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 05:46:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df=spark.range(1,1000000).withColumn("value",col("id")%1000)

In [5]:
df.take(100)

[Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9),
 Row(id=10, value=10),
 Row(id=11, value=11),
 Row(id=12, value=12),
 Row(id=13, value=13),
 Row(id=14, value=14),
 Row(id=15, value=15),
 Row(id=16, value=16),
 Row(id=17, value=17),
 Row(id=18, value=18),
 Row(id=19, value=19),
 Row(id=20, value=20),
 Row(id=21, value=21),
 Row(id=22, value=22),
 Row(id=23, value=23),
 Row(id=24, value=24),
 Row(id=25, value=25),
 Row(id=26, value=26),
 Row(id=27, value=27),
 Row(id=28, value=28),
 Row(id=29, value=29),
 Row(id=30, value=30),
 Row(id=31, value=31),
 Row(id=32, value=32),
 Row(id=33, value=33),
 Row(id=34, value=34),
 Row(id=35, value=35),
 Row(id=36, value=36),
 Row(id=37, value=37),
 Row(id=38, value=38),
 Row(id=39, value=39),
 Row(id=40, value=40),
 Row(id=41, value=41),
 Row(id=42, value=42),
 Row(id=43, value=43),
 Row(id=44, value=44),
 Row(i

In [6]:
print("before partioned: ", df.rdd.getNumPartitions())

df_repartioned = df.repartition(25)
print("after partioned: ", df_repartioned.rdd.getNumPartitions())

before partioned:  2


[Stage 1:>                                                          (0 + 2) / 2]

after partioned:  25


In [7]:
df_coalesced = df_repartioned.coalesce(10)

In [8]:
print("after coalesced: ", df_coalesced.rdd.getNumPartitions())

[Stage 2:>                                                          (0 + 2) / 2]

after coalesced:  10


In [9]:
df_repartioned.write.mode("overwrite").csv("adv_sp_out/demo_data", header=True)

26/06/13 05:46:24 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [10]:
optimized_df = df.filter(col("value")>500).filter(col("id")<500000).select("id","value")

In [11]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#2L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 500000))
   +- *(1) Range (1, 1000000, step=1, splits=2)




In [26]:
start_time = time.time()
count_uncatched = optimized_df.count()
end_time = time.time()
#print(f"1. Optimized Execution | Count: {count_uncached} | Time: (round{end_time - start_time},4) seconds")
print(f"1. optimized execution | count: {count_uncatched} | time: {round(end_time - start_time, 2)} sec")

1. optimized execution | count: 249500 | time: 0.28 sec


In [27]:
optimized_df.cache()

26/06/13 06:16:19 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [28]:
start_time = time.time()
count_first_catched = optimized_df.count()
end_time = time.time()
print(f"2. count first execution | count: {count_first_catched} | time: {round(end_time - start_time, 2)} sec")

2. count first execution | count: 249500 | time: 0.14 sec


In [29]:
start_time = time.time()
count_second_catched = optimized_df.count()
end_time = time.time()
print(f"3. count second execution | count: {count_second_catched} | time: {round(end_time - start_time, 2)} sec")

3. count second execution | count: 249500 | time: 0.12 sec


In [30]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [34]:
data = [("Abhay",23),("Billu",33),("jaggu",17),("surya",19)]
df1 = spark.createDataFrame(data, ["name","age"])

In [35]:
df1.show()

+-----+---+
| name|age|
+-----+---+
|Abhay| 23|
|Billu| 33|
|jaggu| 17|
|surya| 19|
+-----+---+



In [37]:
def can_drive_func(age):
    if age>=18:
        return "can drive with licence"
    return "can drive"

In [41]:
can_drive_udf = udf(can_drive_func, StringType())

# 

In [48]:
df_method1 = df1.withColumn("Category", can_drive_udf(col("age")))
print("function 1: Standard UDF via df API")
df_method1.show()

function 1: Standard UDF via df API


+-----+---+--------------------+
| name|age|            Category|
+-----+---+--------------------+
|Abhay| 23|can drive with li...|
|Billu| 33|can drive with li...|
|jaggu| 17|           can drive|
|surya| 19|can drive with li...|
+-----+---+--------------------+



In [52]:
spark.udf.register("sql_categorize_drive", can_drive_func, StringType())
df1.createOrReplaceTempView("people")

26/06/13 06:59:34 WARN SimpleFunctionRegistry: The function sql_categorize_drive replaced a previously registered function.


In [54]:
sql_df = spark.sql("""
    SELECT 
        name,
        age,
        sql_categorize_drive(age) AS Category
    FROM people
""")
sql_df.show()

+-----+---+--------------------+
| name|age|            Category|
+-----+---+--------------------+
|Abhay| 23|can drive with li...|
|Billu| 33|can drive with li...|
|jaggu| 17|           can drive|
|surya| 19|can drive with li...|
+-----+---+--------------------+



In [55]:
spark.stop()